# Create Arnold and Mabel Beckman Foundation Awards

Creates Arnold and Mabel Beckman Foundation awards from the foundation's own
**Awarded Scientists** directory (https://www.beckman-foundation.org/awarded-scientists/).
Every awardee has a server-rendered profile at `/people/<slug>/` (all listed in
the site's sitemap.xml); each profile carries one block per Beckman award:
Research Title, Beckman Program, Award Year, Institution at Time of Award,
Faculty Mentor, plus the person's name, ORCID (when given) and discipline.
**One row = one award** (a person can hold several, e.g. Beckman Scholar then
Beckman Postdoctoral Fellow).

**2,530 awards to 2,496 people, 1991-2027** (local run 2026-10-01, 2,575
profiles; 79 profiles carry no award block: Scientific Advisory Council,
institute directors, and a few institutional Beckman Scholars program
directors whose award is not itemised). Title 83% (research title), program /
year / institution 100%, ORCID 7%. Beckman Scholars 1,859, Young Investigators
444, Postdoctoral Fellows 178, Speaker & Conference Support 18, center /
instrument grants 24, Argyros 6.

**Flag (kept per the batch scope rule, research-ecosystem support):** the 18
Beckman Speaker and Conference Support Fund awards fund seminars / conferences
(`funder_scheme = 'Beckman Speaker and Conference Support Fund'`); the named
person is often the student organiser.

**Scope:** all Beckman programs in the directory are research funding and are
kept: Beckman Young Investigators, Beckman Scholars (undergraduate research
training), Arnold O. Beckman Postdoctoral Fellows, Beckman-Argyros Award in
Vision Research, Beckman Research Technology Initiative, Beckman Speaker and
Conference Support Fund, and the institutional center/instrument grants
(CryoEM, Light-Sheet Microscopy, FIB-Milling for Cellular CryoET, Mass Spec for
Atmospheric Monitoring). The foundation's Orange County community grants are
not in the directory and are not ingested.

**Prerequisites:**
- Run `scripts/local/beckman_foundation_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/beckman_foundation/beckman_foundation_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** Beckman publishes no grant number. Citing works
write free text (`BYI 2017`, `2021BYI`, `2014-BYI`, `2020Postdoc-0000000038`),
so a stable synthetic key is shipped: `BECKMAN-{program-slug}-{year}-{person-slug}`
(program slug from the program link, person slug from the profile URL). It
collapses onto 0 existing shells (reported in the overlap cell).

**No amounts:** profiles carry no award amount (programs have standard amounts
that changed over time; not back-filled). §6.7 amount check waived; currency NULL.
`start_date` = Jan 1 of the award year (year-only pattern, §2.3); no end date.
`description` NULL: the profile's "Research Summary" describes the person's
current research, not the award.

**Twin:** "Arnold and Mabel Beckman Initiative for Macular Research"
(F4320310357, same ROR 05avmtm72) is a Beckman program (BIMR, 2 shells). Its
awards are not in the directory; nothing is routed to it.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306253`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 05avmtm72, doi 10.13039/100000997)

**Priority:** `561` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.beckman_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/beckman_foundation/beckman_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(DISTINCT person_slug) as people
FROM openalex.awards.beckman_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.beckman_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.beckman_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306253 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306253;

## Step 2: Create Beckman Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.beckman_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306253  -- Arnold and Mabel Beckman Foundation
),

parsed AS (
    SELECT
        g.*,
        TRY_CAST(g.award_year AS INT) AS yr,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(g.family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.given_name), ''),
            'family_name', NULLIF(TRIM(g.family_name), ''),
            'orcid', CASE WHEN g.orcid RLIKE '^[0-9]{4}-' THEN CONCAT('https://orcid.org/', g.orcid) END,
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution_at_award), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.beckman_foundation_raw g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- research title when the profile gives one, else "<program> <year>: <awardee>" (§2.3.1)
    COALESCE(
        NULLIF(TRIM(g.research_title), ''),
        CONCAT_WS(' ', NULLIF(TRIM(g.program), ''), g.award_year) || ': ' ||
            CONCAT_WS(' ', NULLIF(TRIM(g.given_name), ''), NULLIF(TRIM(g.family_name), ''))
    ) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE
        WHEN LOWER(g.program) LIKE '%fellow%' THEN 'fellowship'
        WHEN LOWER(g.program) LIKE '%scholar%' THEN 'training'
        ELSE 'research'
    END as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'beckman_awarded_scientists' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.award_year, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every shipped id is the synthetic BECKMAN-{program}-{year}-{person} key; expect 0 rows.
SELECT funder_award_id, display_name, landing_page_url
FROM openalex.awards.beckman_foundation_awards
WHERE NOT funder_award_id RLIKE '^BECKMAN-[a-z0-9-]+-(19|20)[0-9]{2}-[a-z0-9-]+$';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'beckman_awarded_scientists' AND priority = 561;

-- Insert into openalex_awards_raw with priority.
-- Priority 561: direct-from-funder ingest of the Beckman Foundation's own
-- Awarded Scientists directory. Higher wins under the 2026-06-20 DESC dedup.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    561 as priority
FROM openalex.awards.beckman_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.beckman_foundation_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name
FROM openalex.awards.beckman_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, MIN(start_year) as first_year, MAX(start_year) as last_year
FROM openalex.awards.beckman_foundation_awards
GROUP BY funder_scheme, funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.beckman_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.beckman_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.beckman_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (amount waived: not published per award).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.orcid) as has_pi_orcid,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.beckman_foundation_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement shells (synthetic key: expect 0 matched).
SELECT c.provenance, COUNT(*) as shells, COUNT(t.id) as matched
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.beckman_foundation_awards t ON t.id = c.id
WHERE c.funder_id = 4320306253 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'beckman_awarded_scientists'
GROUP BY provenance, priority;